BIBLIOTECAS

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import StandardScaler
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import  f1_score, recall_score, precision_score, accuracy_score
from sklearn.metrics import classification_report

from sklearn.utils import class_weight

import joblib

LEITURA DOS ARQUIVOS DESCRITORES DE ETNIAS E LÍNGUAS - CLASSIFICAÇÕES PRÉVIAS E ATUAIS

In [ ]:
caminho_arquivo = 'BDD_etnias_20250505.xlsx';
df_etnias_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'BDD_linguas_20240604.xlsx';
df_linguas_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'descritores.xlsx';
df_etnias_novacod = pd.read_excel(caminho_arquivo, sheet_name='etnias');
df_linguas_novacod = pd.read_excel(caminho_arquivo, sheet_name='linguas');

df_ind = pd.read_csv('df_ind_distancia.csv', sep=';', encoding='utf-8-sig')
df_ind

DEFINIÇÃO DE VARIÁVEIS NUMÉRICAS E CATEGÓRICAS

In [ ]:
variaveis_numericas = ['VAL_LATITUDE','VAL_LONGITUDE','cod_setor','distancia']
variaveis_categoricas= ['txt_etnia_entrada_coleta','desc_cod_lingua_1_novacod','tipo_setor', 'CD_TI', 'sobrenome','descricao_mais_proxima']

GARANTIR QUE O DATASET DE TREINO TENHA AO MENOS UM REGISTRO DE CADA CLASSIFICAÇÃO RÓTULO

In [ ]:
# Índices com cardinalidade 1
cardinalidade_etnia = df_ind['desc_cod_etnia_1_novacod'].value_counts()
cardinalidade1_etnia = cardinalidade_etnia[cardinalidade_etnia == 1].index

# Registros com cardinalidade 1
df_cardinalidade1 = df_ind[df_ind['desc_cod_etnia_1_novacod'].isin(cardinalidade1_etnia)].copy()

# Registros com cardinalidade diferente de 1
df_cardinalidadedif1 = df_ind[~df_ind['desc_cod_etnia_1_novacod'].isin(cardinalidade1_etnia)].copy()

# Split treino e validacao_teste
X_treino, X_validacao_teste, y_treino, y_validacao_teste = train_test_split(
    df_cardinalidadedif1.drop(columns=['desc_cod_etnia_1_novacod']),  
    df_cardinalidadedif1['desc_cod_etnia_1_novacod'],
    test_size=0.4,
    stratify=df_cardinalidadedif1['desc_cod_etnia_1_novacod']
)

# Inclusão dos registros de cardinalidade 1 no dataset de treino
X_treino = pd.concat([X_treino, df_cardinalidade1.drop(columns=['desc_cod_etnia_1_novacod'])])
y_treino = pd.concat([y_treino, df_cardinalidade1['desc_cod_etnia_1_novacod']])

# Split validacao e teste
X_teste, X_validacao, y_teste, y_validacao = train_test_split(
    X_validacao_teste,  
    y_validacao_teste,
    test_size=0.5,
)

PIPELINES DE PRÉ-PROCESSAMENTO COM ORDINAL E ONE HOT ENCONDERS

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_pipeline_ordinal = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_pipeline_onehot = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_ordinal = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_ordinal, variaveis_categoricas)
    ]
)

preprocessor_onehot = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_onehot, variaveis_categoricas)
    ]
)

display(preprocessor_ordinal)
display(preprocessor_onehot)

LABEL ENCODER NOS RÓTULOS

In [ ]:
X_treino_encoded = preprocessor_ordinal.fit_transform(X_treino)
X_validacao_encoded = preprocessor_ordinal.transform(X_validacao)
X_teste_encoded = preprocessor_ordinal.transform(X_teste)

label_encoder = LabelEncoder()
y_treino_encoded = label_encoder.fit_transform(y_treino)
y_validacao_encoded = label_encoder.transform(y_validacao)
y_teste_encoded = label_encoder.transform(y_teste)

REDE NEURAL

In [ ]:
embedding_dim = 64
features = 10
vocab_size = int(np.max(X_treino_encoded) + 1)

redeneural = models.Sequential([
    layers.Embedding(input_dim=vocab_size, 
                     output_dim=embedding_dim),
      
    layers.Flatten(),
    
    layers.Dense(1024, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    # Camada de saída com 393 neurônios
    layers.Dense(393, activation='softmax')
])

redeneural.compile(
    #optimizer='adam',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

pesos = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_treino_encoded),
    y=y_treino_encoded
)
pesos_dict = dict(enumerate(pesos))

history = redeneural.fit(
    X_treino_encoded, y_treino_encoded,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    class_weight = pesos_dict,
    verbose=1
)

GRAVAÇÃO DA REDE NEURAL

In [ ]:
joblib.dump(redeneural, 'redeneural.pkl')

MÉTRICAS TREINO

In [ ]:
y_treino_pred_total = redeneural.predict(X_treino_encoded)
y_treino_pred = np.argmax(y_treino_pred_total, axis=1) 

print(classification_report(y_treino_encoded, y_treino_pred))
print(f"Acurácia: {accuracy_score(y_treino_encoded, y_treino_pred):.2%}")
print(f"Precisão: {precision_score(y_treino_encoded, y_treino_pred, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino_encoded, y_treino_pred, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino_encoded, y_treino_pred, average='macro', zero_division=0):.2%}")

MÉTRICAS VALIDAÇÃO

In [ ]:
y_validacao_pred_total = redeneural.predict(X_validacao_encoded)
y_validacao_pred = np.argmax(y_validacao_pred_total, axis=1) 

#print(classification_report(y_validacao_encoded, y_validacao_pred))
print(f"Acurácia: {accuracy_score(y_validacao_encoded, y_validacao_pred):.2%}")
print(f"Precisão: {precision_score(y_validacao_encoded, y_validacao_pred, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_validacao_encoded, y_validacao_pred, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_validacao_encoded, y_validacao_pred, average='macro', zero_division=0):.2%}")

MÉTRICAS TESTE

In [ ]:
y_teste_pred_total = redeneural.predict(X_teste_encoded)
y_teste_pred = np.argmax(y_teste_pred_total, axis=1) 

#print(classification_report(y_teste_encoded, y_teste_pred))
print(f"Acurácia: {accuracy_score(y_teste_encoded, y_teste_pred):.2%}")
print(f"Precisão: {precision_score(y_teste_encoded, y_teste_pred, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_teste_encoded, y_teste_pred, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_teste_encoded, y_teste_pred, average='macro', zero_division=0):.2%}")

TESTE DE PREDIÇÃO

In [ ]:
item = 5512
 
predicao_prob = redeneural.predict(X_validacao_encoded[[item]])
classe_predita_num = np.argmax(predicao_prob)
classe_nome = label_encoder.inverse_transform([classe_predita_num])


print(f"{X_validacao.iloc[item]}\n")
print(f"Real: {y_validacao.iloc[item]}")
print(f"Predição: {classe_nome[0]}")